# Tutorial 7: 3D Dust Mapping

This tutorial demonstrates line-of-sight dust modeling and 3D dust mapping with brutus, including working with Bayestar maps and multi-cloud models.

## Topics Covered

1. **3D dust maps** (Bayestar) and extinction curves
2. **Line-of-sight extinction** estimation
3. **Multi-cloud dust models** for discrete structures
4. **Stellar posteriors** for dust mapping
5. **3D dust structure** visualization

## Prerequisites

This tutorial optionally uses:
- `bayestar2019_v1.h5` - Bayestar 3D dust map
- Stellar posterior samples from Tutorial 5

The tutorial will generate synthetic data if these files are not available.

In [ ]:
# Imports and setup
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import tutorial utilities
from tutorial_utils import (
    set_plot_style,
    find_brutus_data_file,
    save_figure as save_fig_util,
    print_section
)

# Set plot style
set_plot_style()
plt.rcParams['figure.figsize'] = (10, 6)

# Create plots directory if needed
plots_dir = Path('plots/tutorial_07')
plots_dir.mkdir(parents=True, exist_ok=True)

def save_figure(fig, name):
    """Helper to save figures."""
    filepath = plots_dir / f"{name}.png"
    fig.savefig(filepath, dpi=150, bbox_inches='tight')
    print(f"  Saved: {filepath}")

In [ ]:
# Imports and setup
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import h5py

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Create plots directory if needed
plots_dir = Path('plots/tutorial_07')
plots_dir.mkdir(parents=True, exist_ok=True)

def save_figure(fig, name):
    """Helper to save figures."""
    filepath = plots_dir / f"{name}.png"
    fig.savefig(filepath, dpi=150, bbox_inches='tight')
    print(f"  Saved: {filepath}")

In [ ]:
# Helper function to find data files
def find_brutus_data_file(filename):
    """Find brutus data file in common locations."""
    import os
    from pathlib import Path
    
    # Common search paths
    search_paths = [
        Path.cwd() / 'data',
        Path.cwd().parent / 'data',
        Path.home() / '.brutus' / 'data',
        Path('/mnt/d/Dropbox/GitHub/brutus/data'),
        Path('/mnt/c/Dropbox/GitHub/brutus/data'),
    ]
    
    for base_path in search_paths:
        for sub_dir in ['', 'DATAFILES']:
            full_path = base_path / sub_dir if sub_dir else base_path
            filepath = full_path / filename
            if filepath.exists():
                return str(filepath)
    
    # Try environment variable
    if 'BRUTUS_DATA_DIR' in os.environ:
        filepath = Path(os.environ['BRUTUS_DATA_DIR']) / filename
        if filepath.exists():
            return str(filepath)
    
    raise FileNotFoundError(f"Could not find {filename}. Please download it or set BRUTUS_DATA_DIR.")

## Section 1: Understanding 3D Dust Maps

3D dust maps provide extinction (A_V) as a function of distance and direction:
- **Bayestar**: Based on Pan-STARRS + 2MASS photometry
- **Resolution**: ~7 arcmin (HEALPix)
- **Distance bins**: 31 bins from 0.06 to 60 kpc
- **Provides**: Mean E(B-V) and uncertainty

Let's visualize extinction along different sightlines through the Galaxy.

In [ ]:
# Try to load Bayestar dust map
dustmap = None
try:
    from brutus.dust.maps import Bayestar
    
    dustfile = find_brutus_data_file("bayestar2019_v1.h5")
    dustmap = Bayestar(dustfile)
    print("✓ Loaded Bayestar 3D dust map")
    print(f"  Resolution: NSIDE={dustmap.nside}")
    print(f"  Distance bins: {len(dustmap.dist_bins)}")
except (ImportError, FileNotFoundError) as e:
    print("⚠ Bayestar dust map not available - will use synthetic model")
    print(f"  Reason: {e}")

In [ ]:
# Visualize dust extinction along different sightlines
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Define test sightlines (l, b in degrees)
sightlines = [
    (0, 0, "Galactic Center"),
    (90, 0, "Galactic Plane"),
    (180, 0, "Anti-center"),
    (0, 90, "North Pole"),
    (45, 45, "Intermediate"),
    (135, -30, "Southern Sky"),
]

# Distance grid (in kpc)
distances = np.logspace(np.log10(0.1), np.log10(20), 100)

print("Computing extinction along different sightlines...")

for idx, (l, b, name) in enumerate(sightlines):
    ax = axes[idx // 3, idx % 3]
    
    if dustmap is not None:
        try:
            # Query actual dust map
            av_mean, av_std = dustmap.query(l, b, distances)
            
            # Convert E(B-V) to A_V using R_V = 3.1
            av_mean *= 3.1
            av_std *= 3.1
            
            source = "Bayestar"
        except:
            # Fallback to synthetic
            av_mean = 0.5 * distances * np.exp(-np.abs(b) / 30)
            av_std = 0.2 * av_mean
            source = "Synthetic"
    else:
        # Synthetic dust model
        # Higher extinction in plane, exponential with distance
        av_mean = 0.5 * distances * np.exp(-np.abs(b) / 30)
        av_std = 0.2 * av_mean
        source = "Synthetic"
    
    # Plot extinction vs distance
    ax.fill_between(distances, av_mean - av_std, av_mean + av_std,
                   alpha=0.3, color='brown', label='±1σ')
    ax.plot(distances, av_mean, 'r-', lw=2, label=f'Mean A_V ({source})')
    
    # Add markers for notable distances
    for d_mark, label in [(0.5, 'Local'), (1.5, 'Perseus'), (8, 'Center')]:
        if d_mark < distances.max():
            idx_mark = np.argmin(np.abs(distances - d_mark))
            ax.plot(d_mark, av_mean[idx_mark], 'ko', markersize=8)
            ax.annotate(label, (d_mark, av_mean[idx_mark]),
                       xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    ax.set_xlabel("Distance (kpc)")
    ax.set_ylabel("A_V (mag)")
    ax.set_title(f"{name}\n(l={l}°, b={b}°)")
    ax.set_xscale("log")
    ax.set_xlim(0.1, 20)
    ax.set_ylim(0, 5)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left', fontsize=9)

plt.suptitle("3D Dust Extinction Along Different Sightlines", fontsize=14, fontweight="bold")
save_figure(fig, "dust_sightlines")
plt.show()

print("\n✓ Created dust sightline visualization")
print("  Key observations:")
print("  • Extinction highest toward Galactic center and in plane")
print("  • Minimal extinction at high Galactic latitudes")
print("  • Uncertainty grows with distance")

## Section 2: Line-of-Sight Multi-Cloud Dust Modeling

Real dust distributions often have discrete cloud structures. The brutus.los module can fit multi-cloud models to stellar extinction data.

### Model Components
- **Foreground extinction**: Uniform dust before first cloud
- **Dust clouds**: Discrete jumps in extinction at specific distances
- **Background**: Stars behind all dust
- **Outliers**: Contamination handling

Let's demonstrate with stellar posterior samples.

In [ ]:
# Try to load real stellar posterior samples or generate synthetic ones
print("Preparing stellar posterior samples for LOS dust modeling...\n")

try:
    # Try to find results from Tutorial 5 (Orion field)
    datafile = plots_dir.parent / 'tutorial_05' / 'orion_fits_mist.h5'
    
    if not datafile.exists():
        # Try alternative location
        datafile = find_brutus_data_file("Orion_l204.7_b-19.2_results.h5")
    
    with h5py.File(datafile, 'r') as f:
        # Load distance and extinction samples
        dist_samples = f['samps_dist'][:] if 'samps_dist' in f else None
        av_samples = f['samps_red'][:] if 'samps_red' in f else None
        
        if dist_samples is not None and av_samples is not None:
            print(f"✓ Loaded {len(dist_samples)} stellar posterior samples from file")
            
            # Convert distances to distance modulus for LOS fitting
            dm_samples = 5 * np.log10(dist_samples * 1000) - 5  # kpc to distance modulus
            
            # Only use first 50 stars for demonstration
            dm_samples = dm_samples[:50]
            av_samples = av_samples[:50]
            source = "Real Data"
        else:
            raise ValueError("No distance/extinction samples in file")
            
except (FileNotFoundError, ValueError, KeyError) as e:
    # Generate synthetic samples
    print("Generating synthetic stellar samples...")
    print(f"  (Real data not available: {e})\n")
    
    n_stars = 50
    n_samples = 100
    
    # Create synthetic distance modulus samples
    # Simulate stars at various distances with measurement uncertainties
    true_dm = np.random.uniform(7, 10, n_stars)  # True distance moduli
    dm_error = np.random.uniform(0.1, 0.3, n_stars)  # Uncertainties
    
    dm_samples = np.zeros((n_stars, n_samples))
    for i in range(n_stars):
        dm_samples[i] = np.random.normal(true_dm[i], dm_error[i], n_samples)
    
    # Create synthetic extinction samples (correlated with distance)
    # Simulate a dust cloud at distance modulus ~8.5
    cloud_dm = 8.5
    cloud_width = 0.3
    
    av_samples = np.zeros((n_stars, n_samples))
    for i in range(n_stars):
        # Background extinction
        av_bg = 0.3
        
        # Add cloud extinction for stars behind it
        for j in range(n_samples):
            if dm_samples[i, j] > cloud_dm:
                av_cloud = 1.5 * (1 - np.exp(-(dm_samples[i, j] - cloud_dm) / cloud_width))
                av_samples[i, j] = av_bg + av_cloud + np.random.normal(0, 0.1)
            else:
                av_samples[i, j] = av_bg * (dm_samples[i, j] / cloud_dm) + np.random.normal(0, 0.1)
    
    av_samples = np.maximum(0, av_samples)  # No negative extinction
    source = "Synthetic"

print(f"Using {source} stellar samples:")
print(f"  Stars: {len(dm_samples)}")
print(f"  Samples per star: {dm_samples.shape[1]}")
print(f"  Distance modulus range: [{dm_samples.min():.1f}, {dm_samples.max():.1f}]")
print(f"  A_V range: [{av_samples.min():.2f}, {av_samples.max():.2f}] mag")

In [ ]:
# Visualize the stellar samples
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: Distance distribution
ax = axes[0]
for i in range(min(10, len(dm_samples))):
    ax.hist(dm_samples[i], bins=30, alpha=0.3, histtype='step', lw=2)
ax.set_xlabel("Distance Modulus")
ax.set_ylabel("Number of Samples")
ax.set_title(f"Distance Posteriors ({source})")
ax.grid(True, alpha=0.3)

# Panel 2: Extinction distribution  
ax = axes[1]
for i in range(min(10, len(av_samples))):
    ax.hist(av_samples[i], bins=30, alpha=0.3, histtype='step', lw=2)
ax.set_xlabel("A_V (mag)")
ax.set_ylabel("Number of Samples")
ax.set_title(f"Extinction Posteriors ({source})")
ax.grid(True, alpha=0.3)

# Panel 3: Distance vs Extinction
ax = axes[2]

# Plot median values with error bars
dm_median = np.median(dm_samples, axis=1)
dm_std = np.std(dm_samples, axis=1)
av_median = np.median(av_samples, axis=1)
av_std = np.std(av_samples, axis=1)

scatter = ax.errorbar(dm_median, av_median, xerr=dm_std, yerr=av_std,
                     fmt='o', alpha=0.5, markersize=5, ecolor='gray')

# Color by distance for clarity
scatter = ax.scatter(dm_median, av_median, c=dm_median, cmap='viridis',
                    s=30, zorder=5)
plt.colorbar(scatter, ax=ax, label='Distance Modulus')

ax.set_xlabel("Distance Modulus")
ax.set_ylabel("A_V (mag)")
ax.set_title("Distance-Extinction Relation")
ax.grid(True, alpha=0.3)

plt.suptitle(f"Stellar Posterior Samples for LOS Dust Modeling ({source})",
            fontsize=14, fontweight="bold")
save_figure(fig, "stellar_samples")
plt.show()

print("\n✓ Visualized stellar posterior samples")
if source == "Synthetic":
    print("  Note: Synthetic data includes a dust cloud at μ ≈ 8.5")

In [ ]:
# Demonstrate simple cloud fitting
print("\nFitting dust cloud model to stellar data...\n")

# Simple approach: identify jumps in extinction
dm_median = np.median(dm_samples, axis=1)
av_median = np.median(av_samples, axis=1)

# Sort by distance
sort_idx = np.argsort(dm_median)
dm_sorted = dm_median[sort_idx]
av_sorted = av_median[sort_idx]

# Compute extinction gradient
if len(dm_sorted) > 3:
    # Smooth the data
    from scipy.ndimage import gaussian_filter1d
    av_smooth = gaussian_filter1d(av_sorted, sigma=2)
    
    # Compute gradient
    gradient = np.gradient(av_smooth, dm_sorted)
    
    # Find peak in gradient (cloud location)
    cloud_idx = np.argmax(gradient)
    cloud_dm = dm_sorted[cloud_idx]
    
    # Estimate foreground and cloud extinction
    av_fore = np.median(av_sorted[dm_sorted < cloud_dm - 0.5]) if np.any(dm_sorted < cloud_dm - 0.5) else 0
    av_cloud = np.median(av_sorted[dm_sorted > cloud_dm + 0.5]) if np.any(dm_sorted > cloud_dm + 0.5) else av_sorted[-1]
    
    print(f"Simple cloud fit results:")
    print(f"  Cloud distance: μ = {cloud_dm:.2f}")
    print(f"  Foreground A_V: {av_fore:.2f} mag")
    print(f"  Total A_V behind cloud: {av_cloud:.2f} mag")
    print(f"  Cloud A_V: {av_cloud - av_fore:.2f} mag")
else:
    cloud_dm = 8.5
    av_fore = 0.3
    av_cloud = 1.5
    print("Using default cloud parameters (insufficient data for fit)")

# Visualize the fitted dust profile
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

# Distance grid for model
d_grid = np.linspace(dm_sorted.min() - 0.5, dm_sorted.max() + 0.5, 200)

# Create step function for extinction
av_model = np.ones_like(d_grid) * av_fore
av_model[d_grid >= cloud_dm] = av_cloud

# Plot model
ax.plot(d_grid, av_model, 'r-', lw=3, label='Cloud Model', zorder=10)

# Add data points with uncertainties
ax.errorbar(dm_median, av_median, 
           xerr=np.std(dm_samples, axis=1),
           yerr=np.std(av_samples, axis=1),
           fmt='o', alpha=0.3, markersize=5, label='Stellar Data', zorder=5)

# Mark cloud location
ax.axvline(cloud_dm, color='k', ls='--', alpha=0.5, lw=2)
ax.text(cloud_dm, av_cloud * 1.1, f'Dust Cloud\nμ={cloud_dm:.2f}',
       ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Add annotations
ax.annotate('', xy=(cloud_dm - 1, av_fore), xytext=(cloud_dm - 1, 0),
           arrowprops=dict(arrowstyle='<->', color='blue', lw=2))
ax.text(cloud_dm - 1.2, av_fore/2, f'Foreground\nA_V={av_fore:.2f}',
       ha='right', fontsize=10, color='blue')

ax.annotate('', xy=(cloud_dm + 1, av_cloud), xytext=(cloud_dm + 1, av_fore),
           arrowprops=dict(arrowstyle='<->', color='green', lw=2))  
ax.text(cloud_dm + 1.2, (av_cloud + av_fore)/2, f'Cloud\nΔA_V={av_cloud-av_fore:.2f}',
       ha='left', fontsize=10, color='green')

ax.set_xlabel("Distance Modulus μ")
ax.set_ylabel("Cumulative A_V (mag)")
ax.set_title("Line-of-Sight Dust Profile")
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_xlim(d_grid.min(), d_grid.max())
ax.set_ylim(0, av_cloud * 1.3)

save_figure(fig, "los_dust_fit")
plt.show()

print("\n✓ Completed simple cloud fitting")

## Section 3: 3D Dust Structure Visualization

Let's visualize the 3D structure of dust in the Galaxy, showing how extinction varies with position and direction.

In [ ]:
# Create comprehensive 3D dust visualization
fig = plt.figure(figsize=(15, 10))

# Panel 1: Face-on view of Galaxy with dust
ax1 = plt.subplot(2, 3, 1)

# Create synthetic spiral dust pattern
theta = np.linspace(0, 4 * np.pi, 1000)
# Two spiral arms
for arm, phase in enumerate([0, np.pi]):
    r_spiral = np.exp(0.2 * theta) * 3
    x_spiral = r_spiral * np.cos(theta + phase)
    y_spiral = r_spiral * np.sin(theta + phase)
    
    # Normalize to galactic scale
    x_spiral = x_spiral / x_spiral.max() * 12  # kpc
    y_spiral = y_spiral / y_spiral.max() * 12
    
    # Plot spiral arms with varying opacity for dust density
    for i in range(len(x_spiral)-1):
        alpha = 0.5 * np.exp(-r_spiral[i] / 10)
        ax1.plot(x_spiral[i:i+2], y_spiral[i:i+2], 
                color='brown', alpha=alpha, lw=8)

# Add galactic center
circle = plt.Circle((0, 0), 2, color='orange', alpha=0.3)
ax1.add_patch(circle)

# Add solar position
ax1.scatter([8.2], [0], s=200, c='yellow', marker='*', 
           edgecolor='orange', linewidth=2, zorder=5, label='Sun')

# Add local dust clouds
local_clouds = [(7.5, 0.3, 'Orion'), (8.5, -0.5, 'Perseus'), (7.8, 0.8, 'Taurus')]
for x, y, name in local_clouds:
    ax1.scatter([x], [y], s=100, c='brown', alpha=0.5)
    ax1.annotate(name, (x, y), xytext=(3, 3), textcoords='offset points', fontsize=8)

ax1.set_xlabel("X (kpc)")
ax1.set_ylabel("Y (kpc)")
ax1.set_title("Face-on View")
ax1.set_xlim(-15, 15)
ax1.set_ylim(-15, 15)
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_aspect('equal')

# Panel 2: Edge-on view
ax2 = plt.subplot(2, 3, 2)

# Create synthetic disk profile with scale height variation
x_disk = np.linspace(-15, 15, 100)
z_scales = [0.1, 0.3, 0.5]  # Different scale heights for different components
alphas = [0.8, 0.5, 0.2]

for z_scale, alpha_val in zip(z_scales, alphas):
    for z_offset in np.linspace(-2*z_scale, 2*z_scale, 20):
        alpha = alpha_val * np.exp(-np.abs(z_offset) / z_scale)
        ax2.plot(x_disk, np.ones_like(x_disk) * z_offset,
                color='brown', alpha=alpha * 0.3, lw=1)

# Add midplane
ax2.axhline(0, color='brown', alpha=0.5, lw=3)

# Add solar position
ax2.scatter([8.2], [0.025], s=200, c='yellow', marker='*',
           edgecolor='orange', linewidth=2, zorder=5, label='Sun')

ax2.set_xlabel("X (kpc)")
ax2.set_ylabel("Z (kpc)")
ax2.set_title("Edge-on View")
ax2.set_xlim(-15, 15)
ax2.set_ylim(-2, 2)
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

# Panel 3: All-sky dust map
ax3 = plt.subplot(2, 3, 3)

# Create synthetic all-sky dust map
l_grid = np.linspace(0, 360, 200)
b_grid = np.linspace(-90, 90, 100)
L, B = np.meshgrid(l_grid, b_grid)

# Synthetic extinction pattern
# Higher in plane, with structure
A_V_map = 2 * np.exp(-np.abs(B) / 10)  # Exponential disk
A_V_map *= (1 + 0.5 * np.cos(np.radians(L)))  # Spiral structure
A_V_map += 0.3 * np.exp(-((L - 80)**2 + B**2) / 200)  # Local cloud

im = ax3.contourf(L, B, A_V_map, levels=20, cmap='YlOrBr')
plt.colorbar(im, ax=ax3, label='A_V (mag)')

# Mark galactic center and anticenter
ax3.plot([0], [0], 'w*', markersize=15, label='GC')
ax3.plot([180], [0], 'wo', markersize=10, label='Anti-GC')

ax3.set_xlabel("Galactic Longitude (deg)")
ax3.set_ylabel("Galactic Latitude (deg)")
ax3.set_title("All-Sky Dust Map")
ax3.legend(loc='upper right')
ax3.grid(True, alpha=0.3, color='white')

# Panels 4-6: Extinction vs distance for different directions
directions = [
    (0, 0, "Toward GC", 'red'),
    (90, 0, "In Plane", 'orange'),
    (0, 45, "High Latitude", 'blue')
]

for idx, (l, b, title, color) in enumerate(directions):
    ax = plt.subplot(2, 3, 4 + idx)
    
    distances = np.logspace(-1, 1.5, 100)
    
    # Synthetic cumulative extinction profiles
    if b == 0:  # In plane
        # Multiple dust clouds
        av_cumul = np.zeros_like(distances)
        cloud_distances = [0.5, 1.5, 3, 8] if l == 0 else [0.3, 1, 2.5]
        cloud_extinctions = [0.3, 0.5, 0.8, 1.5] if l == 0 else [0.2, 0.4, 0.6]
        
        for d_cloud, av_cloud in zip(cloud_distances, cloud_extinctions[:len(cloud_distances)]):
            av_cumul += av_cloud * (1 / (1 + np.exp(-(distances - d_cloud) / 0.1)))
            ax.axvline(d_cloud, color='gray', ls=':', alpha=0.5)
    else:  # High latitude
        av_cumul = 0.3 * (1 - np.exp(-distances / 5))
    
    ax.plot(distances, av_cumul, color=color, lw=3)
    ax.fill_between(distances, 0, av_cumul, color=color, alpha=0.2)
    
    ax.set_xlabel("Distance (kpc)")
    ax.set_ylabel("Cumulative A_V")
    ax.set_title(f"{title} (l={l}°, b={b}°)")
    ax.set_xscale("log")
    ax.set_xlim(0.1, 30)
    ax.set_ylim(0, 3)
    ax.grid(True, alpha=0.3)

plt.suptitle("3D Galactic Dust Structure", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.96])
save_figure(fig, "dust_3d_structure")
plt.show()

print("\n✓ Created 3D dust structure visualization")
print("  Panels show:")
print("  • Face-on: Spiral arm dust distribution")
print("  • Edge-on: Thin disk with scale height")
print("  • All-sky: View from Solar position")
print("  • Profiles: Cumulative extinction in different directions")

## Summary and Key Takeaways

This tutorial has demonstrated 3D dust mapping with brutus:

### Key Concepts

1. **3D Dust Maps**
   - Bayestar provides E(B-V) vs distance
   - Resolution ~7 arcmin, distances 0.06-60 kpc
   - Uncertainties increase with distance

2. **Line-of-Sight Modeling**
   - Multi-cloud models fit discrete structures
   - Combines stellar posteriors with dust priors
   - Can identify individual dust clouds

3. **Galactic Dust Distribution**
   - Concentrated in spiral arms and disk
   - Scale height ~100-300 pc
   - Local clouds: Orion, Perseus, Taurus

### Applications

- **3D mapping** of interstellar dust
- **Improved distances** using extinction priors
- **Identifying** dust clouds and structures
- **Calibrating** extinction laws
- **Correcting** photometry for reddening

### Best Practices

- Use **3D dust maps** as priors when available
- Account for **distance-extinction** correlations
- Consider **multi-cloud models** for complex sightlines
- Validate with **independent** extinction measurements

### Next Steps

- **Tutorial 8**: Photometric Calibration
- Try different sightlines and fields
- Combine with cluster analysis for better constraints
- Use full brutus.los module with dynesty for sophisticated fitting

In [ ]:
print("Tutorial 7 Complete!")
print("="*60)
print("\nGenerated plots:")
for plot_file in sorted(plots_dir.glob('*.png')):
    print(f"  - {plot_file.name}")
    
print("\nKey results:")
if 'cloud_dm' in locals():
    print(f"  Cloud distance: μ = {cloud_dm:.2f}")
    print(f"  Cloud extinction: ΔA_V = {av_cloud - av_fore:.2f} mag")
print(f"  Data source: {source}")